In [1]:
import sys
import os
notebook_dir = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..', 'lime_ndt')))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))

In [6]:
import numpy as np
from sklearn.metrics import r2_score
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

# ========================
# 4. Fonction pour mesurer la fidélité
# ========================
def explanation_fidelity(explainer, model_regressor, instance_id=0, num_features=10, num_samples=5000, class_id=1):
    np.random.seed(42)
    
    # Expliquer l'instance
    exp = explainer.explain_instance(
        X_test[instance_id],
        predict_fn,
        num_features=num_features,
        model_regressor=model_regressor,
        num_samples=num_samples
    )
    
    # Prédictions locales déjà calculées par LIME/NDT
    y_local = exp.local_pred.flatten()
    
    # Récupérer les indices des perturbations générées
    X_perturb = exp.intercept_features  # ou exp.features selon version
    
    # Comme on n'a pas accès direct aux données, on approxime le R²
    # avec les prédictions locales vs globales sur les mêmes features
    y_global = predict_fn(X_test[instance_id].reshape(1, -1))[:, class_id].repeat(len(y_local))
    
    return r2_score(y_global, y_local)


# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
results["LinearRegression"] = explanation_fidelity(
    explainer_classic, LinearRegression(), instance_id=0, class_id=1
)
results["DecisionTree"] = explanation_fidelity(
    explainer_ndt, DecisionTreeWrapper(), instance_id=0, num_features=X_train.shape[1], class_id=1
)
results["NDT"] = explanation_fidelity(
    explainer_ndt, NDTRegressorWrapper(D=X_train.shape[1], max_depth=3, epochs=10), 
    instance_id=0, num_features=X_train.shape[1], class_id=1
)

# ========================
# 6. Afficher les résultats
# ========================
print("=== Fidélité des explications (R²) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")


AttributeError: 'Explanation' object has no attribute 'intercept_features'